# 실전 09-2: Parent-Child Retriever (문맥 단절 해결)

## 1. 실험 목적
- **문제점**: 검색 정확도를 높이려면 청크(Chunk)를 작게 쪼개야 합니다. 하지만 너무 작게 쪼개면 앞뒤 문맥(Context)이 잘려나가서 LLM이 내용을 엉뚱하게 이해합니다.
- **해결책**: **검색용은 아주 잘게 쪼개고(Child), LLM에게 넘겨줄 원본은 크게 보관(Parent)**합니다. 잘게 쪼개진 조각이 하나라도 검색에 걸리면, 그 조각이 속해있던 거대한 원본 문서를 통째로 LLM에게 넘겨주어 문맥 단절을 완벽히 방지합니다.

In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore

load_dotenv()

C:\Users\USER\AppData\Local\Temp\ipykernel_27708\2114868814.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

## 2. Parent / Child 텍스트 분할기(Splitter) 설계

In [2]:
# 앞서 1번 노트북에서 다운로드 받은 진짜 논문 PDF 재사용
pdf_path = "../data/attention_is_all_you_need.pdf"
loader = PyPDFLoader(pdf_path)
docs = loader.load()

# 1. Parent 분할기: LLM이 읽을 거대한 원본 (2000 글자 단위)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)

# 2. Child 분할기: 검색 엔진이 핀포인트로 찾아낼 아주 작은 조각 (400 글자 단위)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)

# 3. 저장소 세팅
# Vector DB: 쪼개진 자식(Child) 문서들의 임베딩이 저장되는 공간
vectorstore = Chroma(
    collection_name="split_parents", 
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small")
)

# In-Memory Store: 거대한 부모(Parent) 문서들의 원본 텍스트가 저장되는 공간
store = InMemoryStore()

## 3. ParentDocumentRetriever 생성 및 데이터 적재

In [3]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# add_documents()를 호출하면, 랭체인이 알아서 Parent로 먼저 쪼개어 store에 저장하고, 
# 그걸 다시 Child로 쪼개어 vectorstore에 임베딩하여 저장합니다.
print("데이터 적재 중... (10~20초 소요)")
retriever.add_documents(docs)
print("데이터 적재 완료!")

print(f"생성된 Parent 조각 수: {len(list(store.yield_keys()))}")
# Chroma Vector DB에는 수천 개의 Child 조각이 들어가 있습니다.

데이터 적재 중... (10~20초 소요)
데이터 적재 완료!
생성된 Parent 조각 수: 27


## 4. 검색 실험 및 결과 비교

In [4]:
query = "How does the multi-head attention work in this paper?"

# 1. Vector DB에 직접 쿼리 (Child 조각만 가져오기)
# 문맥이 400자밖에 안 돼서 툭툭 끊긴 느낌이 납니다.
sub_docs = vectorstore.similarity_search(query)
print("=== [Child 검색 결과] ===")
print(f"가져온 길이: {len(sub_docs[0].page_content)} 글자")
print(sub_docs[0].page_content)
print("-" * 50)

# 2. ParentDocumentRetriever 로 쿼리
# Vector DB에서 일치하는 Child를 찾은 뒤, 그 Child를 낳은 Parent 전체를 통째로 가져옵니다.
retrieved_docs = retriever.invoke(query)
print("\n=== [Parent 검색 결과] ===")
print(f"가져온 길이: {len(retrieved_docs[0].page_content)} 글자")
print(retrieved_docs[0].page_content)

=== [Child 검색 결과] ===
가져온 길이: 395 글자
output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
Multi-head attention allows the model to jointly attend to information from different representation
subspaces at different positions. With a single attention head, averaging inhibits this.
MultiHead(Q, K, V) = Concat(head1, ...,headh)W O
where headi = Attention(QW Q
i , KWK
--------------------------------------------------

=== [Parent 검색 결과] ===
가져온 길이: 1937 글자
output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
Multi-head attention allows the model to jointly attend to information from different representation
subspaces at different positions. With a single attention head, averaging inhibits this.
MultiHead(Q, K, V) = Concat(head1, ...,headh)W O
where headi = Attention(QW Q
i , KWK
i , V WV
i )
Where the projections are parameter matricesW Q
i ∈ Rdmodel×dk, W K

## 5. 결과 해석

1. **Child 검색 결과**: 400자로 쪼개진 내용만 가져왔기 때문에, 해당 수식이나 개념이 왜 나왔는지 앞뒤 설명이 다 짤려 있습니다. 이 텍스트만 LLM에게 주면 **환각(Hallucination)**을 일으킬 확률이 매우 높습니다.
2. **Parent 검색 결과**: 자식 조각이 포함된 거대한 원본 문단(2000자)을 통째로 가져왔습니다. 덕분에 Multi-Head Attention에 대한 논문의 앞뒤 서론과 결론까지 온전히 확보했습니다.
3. **결론**: 실무에서 PDF 기반 RAG를 만들 때, `ParentDocumentRetriever`는 **정확도(작게 쪼개서 잘 찾음)와 문맥 유지(크게 넘겨줌)라는 두 마리 토끼를 다 잡는 사실상의 표준 아키텍처**입니다.